# Synthetic time series for labs

This notebook generates **reproducible** synthetic data for:

- **Lab 1** — `lab_ts_eda.ipynb` (EDA, decomposition)  
- **Lab 2** — `lab_ts_features.ipynb` (feature engineering)  
- **Lab 3** — `lab_ts_baselines.ipynb` (naive, MA, regression)  
- **Lab 4** — `lab_ts_econometrics.ipynb` (ARIMA / SARIMA / SARIMAX)  
- **Lab 5** — `lab_ts_catboost.ipynb` (CatBoost)  
- **Lab 6** — `lab_ts_deep.ipynb` (RNN / LSTM / GRU)  

The process is:

\[
y_t = T_t + S_t + C_t + \varepsilon_t
\]

where \(T_t\) is **trend**, \(S_t\) is **seasonal** (fixed period), \(C_t\) is optional **slow cycle** (non-integer period), and \(\varepsilon_t\) is **correlated noise** (AR) so autocorrelation structure is realistic.

Outputs are saved under `data/` as CSV files you can import anywhere.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

# --- configuration ---
RNG_SEED = 42
FREQ = "D"  # daily; change to "MS" for month-start monthly, "H" for hourly, etc.
N_PERIODS = 4 * 365  # ~4 years of daily data (good for seasonality + trend)
SEASONAL_PERIOD = 7  # weekly pattern for daily data (use 12 for monthly MS data)

# Output paths
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_MAIN = DATA_DIR / "synthetic_series.csv"
OUT_META = DATA_DIR / "synthetic_series_meta.json"

rng = np.random.default_rng(RNG_SEED)


def make_time_index(n: int, freq: str) -> pd.DatetimeIndex:
    return pd.date_range("2019-01-01", periods=n, freq=freq)


def ar1_noise(n: int, phi: float = 0.35, sigma: float = 1.0, rng: np.random.Generator = rng) -> np.ndarray:
    """AR(1): eps_t = phi * eps_{t-1} + w_t — smoother than i.i.d. Gaussian."""
    w = rng.normal(0.0, sigma, size=n)
    eps = np.zeros(n)
    eps[0] = w[0]
    for t in range(1, n):
        eps[t] = phi * eps[t - 1] + w[t]
    return eps


def generate_components(
    n: int,
    seasonal_period: int,
    rng: np.random.Generator,
) -> dict[str, np.ndarray]:
    t = np.arange(n, dtype=float)

    # Trend: mild linear + gentle quadratic curvature
    trend = 2.0 + 0.0008 * t + 1e-7 * (t - n / 2) ** 2

    # Seasonal: sum of harmonics at fundamental + one overtone (smooth weekly/monthly shape)
    w0 = 2 * np.pi / seasonal_period
    seasonal = (
        4.0 * np.sin(w0 * t + 0.7)
        + 1.5 * np.sin(2 * w0 * t + 0.2)
        + 0.8 * np.cos(3 * w0 * t)
    )

    # Slow "business cycle" — not fixed calendar period (distinguish from seasonality)
    cycle = 2.2 * np.sin(2 * np.pi * t / 380.0 + 1.1)

    noise = ar1_noise(n, phi=0.4, sigma=1.15, rng=rng)

    # Occasional additive outliers (labels for anomaly sections in later labs)
    outliers = np.zeros(n)
    out_idx = rng.choice(n, size=max(1, n // 200), replace=False)
    outliers[out_idx] += rng.normal(0, 6.0, size=len(out_idx))

    y = trend + seasonal + cycle + noise + outliers
    # Shift level so y stays strictly positive (multiplicative decompose / log in later labs)
    offset = float(8.0 - np.min(y))
    if offset > 0:
        trend = trend + offset
        y = y + offset

    return {
        "trend": trend,
        "seasonal": seasonal,
        "cycle": cycle,
        "noise": noise,
        "outliers": outliers,
        "y": y,
    }


def make_exogenous(n: int, rng: np.random.Generator) -> pd.DataFrame:
    """Simple exogenous series for ARIMAX / dynamic regression labs (e.g. 'promo', 'temp')."""
    t = np.arange(n, dtype=float)
    # Smooth "temperature" + noise
    temp = 15 + 8 * np.sin(2 * np.pi * t / 365.0) + rng.normal(0, 0.8, size=n)
    # Step + pulse "promotion" indicator
    promo = np.zeros(n)
    promo[800:830] = 1.0
    promo[1100:1110] = 1.0
    return pd.DataFrame({"temp": temp, "promo": promo})


comps = generate_components(N_PERIODS, SEASONAL_PERIOD, rng)
idx = make_time_index(N_PERIODS, FREQ)
exog = make_exogenous(N_PERIODS, rng)

df = pd.DataFrame(
    {
        "date": idx,
        "y": comps["y"],
        "trend": comps["trend"],
        "seasonal": comps["seasonal"],
        "cycle": comps["cycle"],
        "noise": comps["noise"],
        "outliers": comps["outliers"],
    }
)
df = pd.concat([df, exog], axis=1)
df.to_csv(OUT_MAIN, index=False)

meta = {
    "rng_seed": RNG_SEED,
    "freq": FREQ,
    "n_periods": N_PERIODS,
    "seasonal_period": SEASONAL_PERIOD,
    "columns": {
        "y": "observed series (sum of components + noise + outliers)",
        "trend": "unobserved trend (for teaching / sanity checks)",
        "seasonal": "unobserved seasonal part",
        "cycle": "unobserved slow cycle",
        "noise": "AR(1) noise",
        "outliers": "sparse additive shocks",
        "temp": "exogenous for ARIMAX / regression",
        "promo": "exogenous 0/1 promo",
    },
    "file": str(OUT_MAIN),
}
with open(OUT_META, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"Wrote {OUT_MAIN.resolve()}  shape={df.shape}")
print(f"Meta: {OUT_META}")
df.head()

### How to use in other notebooks

1. Run all cells above once (creates `data/synthetic_series.csv`).
2. In labs, load with `pd.read_csv("data/synthetic_series.csv", parse_dates=["date"])`.
3. Set `SEASONAL_PERIOD` in `generate_data.ipynb` to match your analysis (`7` for weekly-on-daily, `12` for monthly, etc.).

The file includes **hidden components** (`trend`, `seasonal`, …) only for teaching and diagnostics — in real forecasting you only observe `y` (and optional exogenous columns).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df["date"], df["y"], lw=0.8, label="y")
ax.set_title("Synthetic observed series $y_t$")
ax.set_xlabel("time")
ax.legend()
plt.tight_layout()
plt.show()

# Optional: compare to structural parts (normally unknown in practice)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df["date"], df["trend"], label="trend", lw=1.2)
ax.plot(df["date"], df["seasonal"], label="seasonal", alpha=0.8)
ax.plot(df["date"], df["cycle"], label="cycle", alpha=0.8)
ax.set_title("Latent components (for teaching only)")
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()